<a href="https://colab.research.google.com/github/isocan/Pt111-MACE/blob/main/FairChem_UMA_FineTuning_Tutorial_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning Fair-Chem UMA: from DFT labels to a validated MLIP


This notebook teaches the current **Fair-Chem v2 / UMA** workflow. It starts with labelled atomistic structures, creates an ASE-LMDB dataset, generates a Hydra configuration, fine-tunes a pretrained UMA model, and evaluates the result on data that training never saw.

It also explains how this differs from the older OCP/GemNet-OC workflow and turns the iterative strategy in Kwon & Graves (2026) into a reusable active-learning template.

> **Runtime:** choose **Runtime → Change runtime type → GPU**. A modern GPU with at least 16–24 GB is recommended. The small official demo is for plumbing, not for publishing a potential.

### Learning objectives

By the end, you should be able to:

1. choose an appropriate UMA task and regression target;
2. construct leakage-resistant train/validation/test splits;
3. audit DFT labels, units, constraints, and theory consistency;
4. generate Fair-Chem's current fine-tuning configuration;
5. run and resume a Hydra training job;
6. compare zero-shot and fine-tuned models on held-out structures;
7. validate the potential as a physical model, not just a low test MAE;
8. design an iterative MD → diversity selection → DFT → fine-tune loop.

### What this notebook deliberately does not hide

- Fine-tuning UMA replaces the pretrained output heads with newly initialized heads while retaining the pretrained backbone. Therefore, “step 0” of a fine-tune is **not** the same predictor as the downloaded zero-shot model.
- The official dataset converter currently requires both **energy and forces** on every frame, even if the requested regression task is energy-only.
- A randomly split test set can be badly optimistic for correlated MD or relaxation trajectories.
- A low IID test error does not establish stable molecular dynamics, correct reaction energetics, or safe extrapolation.

## 0. Version contract and trusted inputs

Fair-Chem v2 is a breaking change from legacy OCP/Fair-Chem v1. This tutorial pins the repository to the exact source revision inspected while preparing it:

`e4425dcbd6649b36a82bc513585b8664fb7a6b0a` (2026-08-21)

The pin matters because model names, tasks, CLI arguments, and configs can change. In particular, the current source uses `--regression-tasks` (plural), even though one rendered documentation example has shown the singular spelling.

**Security:** Hydra configuration files can instantiate Python objects. Only run configurations from a repository or person you trust. Do not execute an arbitrary YAML file downloaded from the internet.

In [1]:
from pathlib import Path
import os, sys, subprocess, json, shutil, math, time

FAIRCHEM_REF = "e4425dcbd6649b36a82bc513585b8664fb7a6b0a"
REPO = Path("/content/fairchem")
WORK = Path("/content/uma_finetune_tutorial")
WORK.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("Working directory:", WORK)

Python: 3.13.15
Working directory: /content/uma_finetune_tutorial


## 1. Install the pinned Fair-Chem source

The current editable-install path is `packages/fairchem-core`, not the legacy `src/packages/...` path. Restart the Colab runtime if pip asks you to do so, then continue from the imports cell.

In [2]:
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "https://github.com/facebookresearch/fairchem.git", str(REPO)],
        check=True,
    )

subprocess.run(["git", "-C", str(REPO), "fetch", "origin", FAIRCHEM_REF, "--depth", "1"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", FAIRCHEM_REF], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}/packages/fairchem-core[dev]"],
    check=True,
)

print("Checked out:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip())

Checked out: e4425dcbd6649b36a82bc513585b8664fb7a6b0a


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import yaml

from ase import Atoms
from ase.constraints import FixAtoms
from ase.io import read, write
from ase.calculators.singlepoint import SinglePointCalculator

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

torch: 2.13.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory (GiB): 14.6


## 2. The mental model: what exactly is being fine-tuned?

UMA is a shared equivariant backbone trained across multiple atomistic domains, with task-specific prediction heads. In the standard Fair-Chem fine-tuning path:

- the **backbone** begins from pretrained weights;
- existing pretrained heads are removed;
- a **new head** for your chosen task and targets is initialized;
- all trainable weights are optimized on your dataset.

This explains an initially surprising observation: a newly initialized fine-tuning job can perform worse at step 0 than the original zero-shot checkpoint. Judge it after training and on a truly held-out set.

### Choose the domain head by physics, not by the file format

| Task | Typical domain | Good starting point |
|---|---|---|
| `oc20` | adsorbates and catalytic surfaces | conventional heterogeneous catalysis |
| `oc25` | electro/catalysis | electrochemical interfaces and related surfaces |
| `omat` | bulk inorganic materials | crystals, defects, phonons, elastic properties |
| `omol` | molecules and molecular systems | gas-phase/organic chemistry |
| `odac` | metal-organic frameworks | adsorption in porous frameworks |
| `omc` | molecular crystals | molecular solids |

Task availability can depend on the installed UMA release. For Pt(111) O/OH/OOH work, start by comparing `oc20` and `oc25` zero-shot predictions against a small DFT validation set; use `oc25` if the target data represent electro/catalysis. Do not mix incompatible DFT levels and expect the model to infer which theory produced each label.

### Choose the target

- `e`: energy loss; labels still need forces for the current converter.
- `ef`: energy + forces; usual choice for relaxation and MD.
- `efs`: energy + forces + stress; use for cell/strain/barostat work only when stress labels are reliable and consistently signed/ordered.

## 3. Configure a smoke test or your real project

The official demo verifies the software path quickly. It is intentionally not a scientifically adequate fine-tune. For your own catalysis project, set `USE_OFFICIAL_DEMO=False`, place labelled files in three separate directories, and use `oc20` or `oc25` with `ef`.

Recommended progression:

1. official demo, 1 epoch, small neighbor count;
2. your data, short smoke run;
3. scientifically chosen hyperparameters and an untouched test set;
4. physical validation and active learning.

In [4]:
# ---------- Edit this cell ----------
USE_OFFICIAL_DEMO = True

if USE_OFFICIAL_DEMO:
    BASE_MODEL = "uma-s-1p2"
    UMA_TASK = "omat"
    REGRESSION_TASKS = "e"
    RAW_TRAIN = REPO / "docs/core/common_tasks/finetune_assets/train"
    RAW_VAL = REPO / "docs/core/common_tasks/finetune_assets/val"
    RAW_TEST = RAW_VAL  # plumbing only; NOT an independent scientific test
else:
    BASE_MODEL = "uma-s-1p2"
    UMA_TASK = "oc25"       # consider "oc20" for conventional surface catalysis
    REGRESSION_TASKS = "ef" # use "efs" only with trustworthy stress labels
    RAW_TRAIN = WORK / "raw/train"
    RAW_VAL = WORK / "raw/val"
    RAW_TEST = WORK / "raw/test"  # never pass this directory to the trainer

DATASET_TAG = "demo_v1" if USE_OFFICIAL_DEMO else "my_project_v1"
DATASET_OUT = WORK / f"aselmdb_{DATASET_TAG}"
RUN_ID = "uma_ft_smoke_v1"
RUN_DIR = WORK / "runs"

# Expensive/gated operations are opt-in so every teaching cell is safe to inspect.
RUN_ZERO_SHOT = False
RUN_TRAINING = False
RUN_FINE_TUNED_EVAL = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print({
    "base_model": BASE_MODEL,
    "task": UMA_TASK,
    "targets": REGRESSION_TASKS,
    "device": DEVICE,
    "train": str(RAW_TRAIN),
    "val": str(RAW_VAL),
    "test": str(RAW_TEST),
})

{'base_model': 'uma-s-1p2', 'task': 'omat', 'targets': 'e', 'device': 'cuda', 'train': '/content/fairchem/docs/core/common_tasks/finetune_assets/train', 'val': '/content/fairchem/docs/core/common_tasks/finetune_assets/val', 'test': '/content/fairchem/docs/core/common_tasks/finetune_assets/val'}


### Hugging Face model access

UMA checkpoints are gated. Accept the model's terms on Hugging Face first, then run this login cell. A token entered by the widget is preferable to pasting a secret into notebook source.

In [5]:
DO_HF_LOGIN = False
if DO_HF_LOGIN:
    from huggingface_hub import notebook_login
    notebook_login()
else:
    print("Set DO_HF_LOGIN=True after obtaining model access. Never hard-code a token in the notebook.")

Set DO_HF_LOGIN=True after obtaining model access. Never hard-code a token in the notebook.


## 4. Build labels correctly

Fair-Chem reads ASE `Atoms` objects with a calculator containing reference results. A robust portable pattern is to attach a `SinglePointCalculator` and write `.traj` files.

### Unit contract

- energy: eV per structure;
- forces: eV/Å, shape `(N, 3)`;
- stress: eV/Å³ in ASE convention;
- positions/cell: Å.

For periodic systems, preserve the cell and PBC. Preserve `FixAtoms` constraints when they describe the intended simulation; Fair-Chem's data statistics exclude fixed atoms from the force RMS. Check the sign and Voigt ordering of stress exported by your DFT code before using `efs`.

In [6]:
def attach_reference_labels(atoms, energy, forces, stress=None):
    '''Return a copy carrying immutable DFT labels in ASE's standard calculator interface.'''
    labelled = atoms.copy()
    kwargs = {
        "energy": float(energy),
        "forces": np.asarray(forces, dtype=float),
    }
    if stress is not None:
        kwargs["stress"] = np.asarray(stress, dtype=float)
    labelled.calc = SinglePointCalculator(labelled, **kwargs)
    return labelled


def export_last_dft_frame(source, destination):
    '''Example: convert the last ASE-readable VASP/QE frame to a labelled .traj file.'''
    atoms = read(source, index=-1)
    energy = atoms.get_potential_energy()
    forces = atoms.get_forces(apply_constraint=False)
    stress = None
    if atoms.calc is not None and "stress" in getattr(atoms.calc, "results", {}):
        stress = atoms.get_stress()  # ASE Voigt order, eV/Å^3
    labelled = attach_reference_labels(atoms, energy, forces, stress)
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    write(destination, labelled)
    return destination


# Example only:
# export_last_dft_frame("vasprun.xml", WORK / "raw/train/system_0001.traj")

### Split before training: prevent trajectory leakage

Do not randomly split individual frames from the same MD trajectory or relaxation. Adjacent frames can be nearly duplicates, so an apparently excellent test set may measure memorization.

Split by an independent group such as:

- trajectory or AIMD seed;
- parent material/composition;
- surface facet or adsorbate family;
- defect type;
- active-learning round;
- temperature/pressure regime.

For an iterative workflow, the strongest test is often the **next round of configurations generated by the current model**, labelled afterward by DFT and kept unseen until evaluation.

In [7]:
# Example group-aware split for a metadata table with one row per file.
from sklearn.model_selection import GroupShuffleSplit

def group_split(metadata, group_column="trajectory_id", seed=7):
    '''Return train/validation/test dataframes without splitting a group across sets.'''
    first = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
    train_idx, hold_idx = next(first.split(metadata, groups=metadata[group_column]))
    train = metadata.iloc[train_idx].copy()
    hold = metadata.iloc[hold_idx].copy()

    second = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=seed + 1)
    val_rel, test_rel = next(second.split(hold, groups=hold[group_column]))
    return train, hold.iloc[val_rel].copy(), hold.iloc[test_rel].copy()


# metadata = pd.DataFrame({"path": [...], "trajectory_id": [...]})
# train_meta, val_meta, test_meta = group_split(metadata)

## 5. Audit the raw structures before conversion

The converter recursively scans every file in each directory. Keep only ASE-readable structure files there—no plots, logs, checkpoints, or YAML files.

The audit below checks the label interface, array shapes, finite values, cell/PBC consistency, minimum interatomic distances, elements, and constraints. It does not replace chemical judgment.

In [8]:
SUPPORTED_SUFFIXES = {".traj", ".xyz", ".extxyz", ".cif", ".vasp", ".poscar", ".json", ".db"}

def structure_files(folder):
    folder = Path(folder)
    if not folder.exists():
        raise FileNotFoundError(folder)
    files = [p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in SUPPORTED_SUFFIXES]
    if not files:
        raise FileNotFoundError(f"No supported structure files found under {folder}")
    return sorted(files)


def iter_frames(folder):
    for path in structure_files(folder):
        images = read(path, index=":")
        if isinstance(images, Atoms):
            images = [images]
        for frame_index, atoms in enumerate(images):
            yield path, frame_index, atoms


def free_atom_mask(atoms):
    mask = np.ones(len(atoms), dtype=bool)
    for constraint in atoms.constraints:
        if isinstance(constraint, FixAtoms):
            mask[np.asarray(constraint.get_indices(), dtype=int)] = False
    return mask


def minimum_distance(atoms):
    if len(atoms) < 2:
        return np.nan
    distances = atoms.get_all_distances(mic=bool(np.any(atoms.pbc)))
    distances[distances == 0.0] = np.inf
    return float(np.min(distances))


def audit_folder(folder, require_stress=False):
    rows, errors = [], []
    for path, frame_index, atoms in iter_frames(folder):
        try:
            if atoms.calc is None:
                raise ValueError("no attached calculator/reference labels")
            energy = float(atoms.get_potential_energy())
            forces = np.asarray(atoms.get_forces(apply_constraint=False))
            if forces.shape != (len(atoms), 3):
                raise ValueError(f"force shape {forces.shape}, expected {(len(atoms), 3)}")
            if not np.isfinite(energy) or not np.all(np.isfinite(forces)):
                raise ValueError("non-finite energy or force")
            has_stress = "stress" in getattr(atoms.calc, "results", {})
            if require_stress and not has_stress:
                raise ValueError("stress required but absent")
            if require_stress and not np.all(np.isfinite(atoms.get_stress())):
                raise ValueError("non-finite stress")
            rows.append({
                "file": path.name,
                "frame": frame_index,
                "n_atoms": len(atoms),
                "formula": atoms.get_chemical_formula(),
                "energy_eV": energy,
                "energy_eV_per_atom": energy / len(atoms),
                "force_rms_eV_A": float(np.sqrt(np.mean(forces[free_atom_mask(atoms)] ** 2))),
                "max_force_eV_A": float(np.max(np.linalg.norm(forces[free_atom_mask(atoms)], axis=1))),
                "min_distance_A": minimum_distance(atoms),
                "n_fixed": int((~free_atom_mask(atoms)).sum()),
                "periodic": bool(np.any(atoms.pbc)),
                "has_stress": has_stress,
            })
        except Exception as exc:
            errors.append(f"{path.name}[{frame_index}]: {exc}")
    if errors:
        raise ValueError("Dataset audit failed:\n  " + "\n  ".join(errors[:20]))
    return pd.DataFrame(rows)


require_stress = REGRESSION_TASKS == "efs"
train_audit = audit_folder(RAW_TRAIN, require_stress=require_stress)
val_audit = audit_folder(RAW_VAL, require_stress=require_stress)
test_audit = audit_folder(RAW_TEST, require_stress=require_stress)

display(pd.DataFrame({
    "split": ["train", "validation", "test"],
    "frames": [len(train_audit), len(val_audit), len(test_audit)],
    "atoms": [train_audit.n_atoms.sum(), val_audit.n_atoms.sum(), test_audit.n_atoms.sum()],
    "min_distance_A": [train_audit.min_distance_A.min(), val_audit.min_distance_A.min(), test_audit.min_distance_A.min()],
    "max_force_eV_A": [train_audit.max_force_eV_A.max(), val_audit.max_force_eV_A.max(), test_audit.max_force_eV_A.max()],
}))
display(train_audit.head())

,split,frames,atoms,min_distance_A,max_force_eV_A
0,train,2,48,2.555359,5.105201
1,validation,2,64,1.806282,6.054072
2,test,2,64,1.806282,6.054072


,file,frame,n_atoms,formula,energy_eV,energy_eV_per_atom,force_rms_eV_A,max_force_eV_A,min_distance_A,n_fixed,periodic,has_stress
0,structure_0001.traj,0,32,Cr32,-66.649775,-2.082805,1.593313,5.105201,2.555359,0,True,False
1,structure_0002.traj,0,16,Ni16,-52.102479,-3.256405,1.849710,4.762043,2.662490,0,True,False


### Data review questions for a class

Before continuing, ask:

1. Are all labels from the same exchange-correlation functional, pseudopotentials, cutoffs, spin treatment, dispersion correction, and `+U` convention?
2. Does the train set span the chemistry and geometry that will be visited during inference?
3. Are extreme force frames physically meaningful, or failed electronic-structure calculations?
4. Does the dataset include compressed contacts and transition-like configurations needed for stable MD?
5. Is the test set independent by trajectory/system/active-learning round?

The converter fits elemental linear energy references on the training split and calculates the training force RMS. This is especially important when your target theory differs from UMA pretraining.

## 6. Optional: establish the zero-shot baseline

The baseline answers a crucial question: did fine-tuning improve the target distribution relative to the released model? It also helps compare candidate domain heads (`oc20` versus `oc25`, for example).

Set `RUN_ZERO_SHOT=True` only after GPU and Hugging Face access are ready. The evaluation function preserves DFT labels before attaching the MLIP calculator and reports energy error per atom plus both force-component and force-vector errors on free atoms.

In [9]:
def evaluate_calculator(calculator, folder, max_frames=None):
    rows, force_ref_all, force_pred_all = [], [], []
    for count, (path, frame_index, labelled) in enumerate(iter_frames(folder)):
        if max_frames is not None and count >= max_frames:
            break

        ref_e = float(labelled.get_potential_energy())
        ref_f = np.asarray(labelled.get_forces(apply_constraint=False)).copy()
        mask = free_atom_mask(labelled)

        probe = labelled.copy()
        probe.calc = calculator
        pred_e = float(probe.get_potential_energy())
        pred_f = np.asarray(probe.get_forces(apply_constraint=False))

        delta_f = pred_f[mask] - ref_f[mask]
        rows.append({
            "file": path.name,
            "frame": frame_index,
            "n_atoms": len(labelled),
            "ref_E_per_atom": ref_e / len(labelled),
            "pred_E_per_atom": pred_e / len(labelled),
            "abs_E_error_meV_per_atom": 1000 * abs(pred_e - ref_e) / len(labelled),
            "force_component_MAE_eV_A": float(np.mean(np.abs(delta_f))),
            "force_vector_MAE_eV_A": float(np.mean(np.linalg.norm(delta_f, axis=1))),
            "max_force_vector_error_eV_A": float(np.max(np.linalg.norm(delta_f, axis=1))),
        })
        force_ref_all.append(ref_f[mask].reshape(-1))
        force_pred_all.append(pred_f[mask].reshape(-1))

    frame_table = pd.DataFrame(rows)
    summary = {
        "frames": len(frame_table),
        "energy_MAE_meV_per_atom": frame_table.abs_E_error_meV_per_atom.mean(),
        "force_component_MAE_eV_A": frame_table.force_component_MAE_eV_A.mean(),
        "force_vector_MAE_eV_A": frame_table.force_vector_MAE_eV_A.mean(),
        "max_force_vector_error_eV_A": frame_table.max_force_vector_error_eV_A.max(),
    }
    arrays = {
        "force_ref": np.concatenate(force_ref_all),
        "force_pred": np.concatenate(force_pred_all),
    }
    return frame_table, summary, arrays


def parity_plots(frame_table, arrays, title):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    panels = [
        (frame_table.ref_E_per_atom, frame_table.pred_E_per_atom, "Energy (eV/atom)"),
        (arrays["force_ref"], arrays["force_pred"], "Force component (eV/Å)"),
    ]
    for ax, (reference, prediction, label) in zip(axes, panels):
        reference, prediction = np.asarray(reference), np.asarray(prediction)
        lo, hi = min(reference.min(), prediction.min()), max(reference.max(), prediction.max())
        ax.scatter(reference, prediction, s=16, alpha=0.6)
        ax.plot([lo, hi], [lo, hi], "k--", lw=1)
        ax.set(xlabel=f"DFT {label}", ylabel=f"MLIP {label}")
    fig.suptitle(title)
    fig.tight_layout()
    return fig

In [10]:
zero_shot_results = None

if RUN_ZERO_SHOT:
    if not torch.cuda.is_available():
        raise RuntimeError("A GPU is strongly recommended for UMA inference.")
    from fairchem.core import FAIRChemCalculator

    zero_calc = FAIRChemCalculator.from_model_checkpoint(
        BASE_MODEL,
        task_name=UMA_TASK,
        device=DEVICE,
    )
    zero_table, zero_summary, zero_arrays = evaluate_calculator(zero_calc, RAW_TEST)
    zero_shot_results = zero_summary
    print(json.dumps(zero_summary, indent=2))
    parity_plots(zero_table, zero_arrays, f"Zero-shot {BASE_MODEL} / {UMA_TASK}")
    plt.show()
    del zero_calc
    torch.cuda.empty_cache()
else:
    print("Skipped. Set RUN_ZERO_SHOT=True after model access is configured.")

Skipped. Set RUN_ZERO_SHOT=True after model access is configured.


## 7. Convert raw files to ASE-LMDB and generate the Hydra config

The official helper performs four coupled operations:

1. reads all train/validation structures through ASE;
2. writes ASE-LMDB datasets;
3. computes force-RMS normalization from the training data;
4. fits element-wise linear energy references and generates the UMA data/training YAML.

It requires `DATASET_OUT` not to exist on first creation. Change `DATASET_TAG` for a new dataset version. Set `REBUILD_DATASET=True` only when you intentionally want to replace that narrow generated directory.

In [11]:
REBUILD_DATASET = False

if DATASET_OUT.exists() and REBUILD_DATASET:
    # Scoped to the explicitly configured generated dataset directory.
    shutil.rmtree(DATASET_OUT)

converter = REPO / "src/fairchem/core/scripts/create_uma_finetune_dataset.py"
convert_cmd = [
    sys.executable,
    str(converter),
    "--train-dir", str(RAW_TRAIN),
    "--val-dir", str(RAW_VAL),
    "--output-dir", str(DATASET_OUT),
    "--uma-task", UMA_TASK,
    "--regression-tasks", REGRESSION_TASKS,  # plural in the current source
    "--base-model", BASE_MODEL,
    "--num-workers", "2",
]

print("Command:\n", " \\\n+  ".join(convert_cmd))
if not DATASET_OUT.exists():
    # The converter resolves its template files relative to the repository root.
    subprocess.run(convert_cmd, check=True, cwd=REPO)
else:
    print("Reusing", DATASET_OUT, "— change DATASET_TAG or set REBUILD_DATASET=True to regenerate.")

Command:
 /usr/bin/python3 \
+  /content/fairchem/src/fairchem/core/scripts/create_uma_finetune_dataset.py \
+  --train-dir \
+  /content/fairchem/docs/core/common_tasks/finetune_assets/train \
+  --val-dir \
+  /content/fairchem/docs/core/common_tasks/finetune_assets/val \
+  --output-dir \
+  /content/uma_finetune_tutorial/aselmdb_demo_v1 \
+  --uma-task \
+  omat \
+  --regression-tasks \
+  e \
+  --base-model \
+  uma-s-1p2 \
+  --num-workers \
+  2


In [12]:
generated_yamls = sorted(DATASET_OUT.glob("*.yaml"))
if not generated_yamls:
    raise FileNotFoundError(f"No generated YAML found in {DATASET_OUT}")

print("Generated files:")
for path in sorted(DATASET_OUT.rglob("*")):
    if path.is_file():
        print(" -", path.relative_to(DATASET_OUT))

for path in generated_yamls:
    print("\n###", path.name)
    print(path.read_text()[:8000])

Generated files:
 - data/uma_conserving_data_task_energy.yaml
 - train/data.0000.aselmdb
 - train/data.0000.aselmdb-lock
 - train/data.0000.failed
 - train/data.0000.log
 - train/data.0001.aselmdb
 - train/data.0001.aselmdb-lock
 - train/data.0001.failed
 - train/data.0001.log
 - train/metadata.npz
 - uma_sm_finetune_template.yaml
 - val/data.0000.aselmdb
 - val/data.0000.aselmdb-lock
 - val/data.0000.failed
 - val/data.0000.log
 - val/data.0001.aselmdb
 - val/data.0001.aselmdb-lock
 - val/data.0001.failed
 - val/data.0001.log
 - val/metadata.npz

### uma_sm_finetune_template.yaml
defaults:
- data: uma_conserving_data_task_energy
- _self_
job:
  device_type: CUDA
  scheduler:
    mode: LOCAL
    ranks_per_node: 1
    num_nodes: 1
  debug: true
  run_dir: /tmp/uma_finetune_runs/
  run_name: uma_finetune
  logger:
    _target_: fairchem.core.common.logger.WandBSingletonLogger.init_wandb
    _partial_: true
    entity: example
    project: uma_finetune
base_model_name: uma-s-1p2
max_neigh

### Inspect, do not blindly trust, generated normalization

Confirm that:

- train and validation paths point to the expected ASE-LMDB directories;
- `base_model_name`, task, and regression targets are correct;
- energy references include all elements in validation/test/inference;
- force RMS is finite and plausible;
- target coefficients match your scientific priority.

The standard energy-force config weights per-atom energy MAE and force loss differently. Loss coefficients set optimization priorities; they are not physical units or reported error metrics.

In [ ]:
def yaml_overview(path):
    # safe_load is appropriate for inspection; do not use untrusted Hydra configs for execution.
    cfg = yaml.safe_load(Path(path).read_text())
    keys = ["base_model_name", "epochs", "batch_size", "lr", "max_neighbors", "weight_decay"]
    return {key: cfg.get(key, "<nested or absent>") for key in keys}

for path in generated_yamls:
    print(path.name, yaml_overview(path))

## 8. Launch a short training run

Start with a smoke test that proves data loading, forward/backward passes, validation, and checkpoint writing. Then increase the scientifically important settings.

Teaching defaults below:

- `batch_size=1` to reduce memory pressure;
- `max_neighbors=50` only for a plumbing test—dense structures may need 100–300;
- 2 epochs for the demo;
- `job.debug=True` to avoid requiring Weights & Biases.

For a real run, monitor both train and validation curves. Use a lower learning rate and/or warmup when a small dataset is unstable. Too few neighbors silently changes the local environment; it is not merely a speed setting.

In [13]:
TRAIN_EPOCHS = 2 if USE_OFFICIAL_DEMO else 20
BATCH_SIZE = 1
MAX_NEIGHBORS = 50 if USE_OFFICIAL_DEMO else 100
LEARNING_RATE = 2e-4

# The generator creates a fine-tuning template; choose the filename containing "finetune".
training_yaml_candidates = [p for p in generated_yamls if "finetune" in p.name.lower()]
TRAINING_YAML = training_yaml_candidates[0] if training_yaml_candidates else generated_yamls[0]

train_cmd = [
    "fairchem", "-c", str(TRAINING_YAML),
    f"job.run_dir={RUN_DIR}",
    f"+job.timestamp_id={RUN_ID}",
    "job.debug=True",
    f"epochs={TRAIN_EPOCHS}",
    "steps=null",
    f"batch_size={BATCH_SIZE}",
    f"lr={LEARNING_RATE}",
    f"max_neighbors={MAX_NEIGHBORS}",
    "evaluate_every_n_steps=10",
    "checkpoint_every_n_steps=100",
]

print("Training command:\n", " \\\n+  ".join(train_cmd))

if RUN_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a GPU runtime before training UMA.")
    subprocess.run(train_cmd, check=True)
else:
    print("Dry run only. Set RUN_TRAINING=True when data, GPU, and model access are ready.")

Training command:
 fairchem \
+  -c \
+  /content/uma_finetune_tutorial/aselmdb_demo_v1/uma_sm_finetune_template.yaml \
+  job.run_dir=/content/uma_finetune_tutorial/runs \
+  +job.timestamp_id=uma_ft_smoke_v1 \
+  job.debug=True \
+  epochs=2 \
+  steps=null \
+  batch_size=1 \
+  lr=0.0002 \
+  max_neighbors=50 \
+  evaluate_every_n_steps=10 \
+  checkpoint_every_n_steps=100
Dry run only. Set RUN_TRAINING=True when data, GPU, and model access are ready.


### Production tuning guide

Do not optimize hyperparameters against the test set.

| Symptom | Diagnose first | Possible response |
|---|---|---|
| train and validation both high | insufficient coverage, wrong task/theory, labels | improve data; verify labels and elemental coverage |
| train falls, validation rises | overfitting or split shift | early stop, more diverse data, lower capacity/update magnitude |
| NaN/divergence | bad labels, extreme geometries, LR, precision | audit outliers; lower LR; add warmup; inspect gradients |
| good energy, poor forces | loss balance or inadequate local environments | increase force priority; add non-equilibrium structures |
| good IID metrics, unstable MD | coverage holes/short contacts | active learning; repulsive/high-force configurations |
| worse zero-shot behavior outside target | catastrophic forgetting/domain specialization | evaluate OOD; replay data or retain separate models |

The attached WS₂ oxygen-plasma study used 55 epochs, cosine scheduling, 5 warmup epochs, EMA, and energy/force/stress losses—but those are research-specific choices, not universal defaults.

## 9. Locate, resume, and load the checkpoint

The standard final inference checkpoint is written under:

`RUN_DIR / RUN_ID / checkpoints / final / inference_ckpt.pt`

A `resume.yaml` is stored with the checkpoint. Use it to resume the same job; do not reconstruct the full resolved configuration by memory.

In [14]:
FINAL_DIR = RUN_DIR / RUN_ID / "checkpoints/final"
INFERENCE_CKPT = FINAL_DIR / "inference_ckpt.pt"
RESUME_YAML = FINAL_DIR / "resume.yaml"

print("Expected checkpoint:", INFERENCE_CKPT)
print("Checkpoint exists:", INFERENCE_CKPT.exists())
print("Resume config exists:", RESUME_YAML.exists())

if RESUME_YAML.exists():
    print("Resume with:")
    print("fairchem -c", RESUME_YAML)

Expected checkpoint: /content/uma_finetune_tutorial/runs/uma_ft_smoke_v1/checkpoints/final/inference_ckpt.pt
Checkpoint exists: False
Resume config exists: False


## 10. Evaluate the fine-tuned model on the untouched test split

Use exactly the same metrics and frames as the zero-shot baseline. Report the distribution (parity/residual plots and worst cases), not just one mean.

For adsorption science, also evaluate relevant **energy differences** using a consistent reference scheme. A small total-energy MAE does not guarantee accurate adsorption energies because errors in the slab, adsorbate slab, and molecular reference can combine.

In [15]:
fine_tuned_results = None

if RUN_FINE_TUNED_EVAL:
    if not INFERENCE_CKPT.exists():
        raise FileNotFoundError(INFERENCE_CKPT)
    from fairchem.core import FAIRChemCalculator

    ft_calc = FAIRChemCalculator.from_model_checkpoint(
        str(INFERENCE_CKPT),
        task_name=UMA_TASK,
        device=DEVICE,
    )
    ft_table, ft_summary, ft_arrays = evaluate_calculator(ft_calc, RAW_TEST)
    fine_tuned_results = ft_summary
    print(json.dumps(ft_summary, indent=2))
    parity_plots(ft_table, ft_arrays, f"Fine-tuned {BASE_MODEL} / {UMA_TASK}")
    plt.show()

    if zero_shot_results is not None:
        comparison = pd.DataFrame([zero_shot_results, fine_tuned_results], index=["zero-shot", "fine-tuned"])
        display(comparison)

    worst = ft_table.sort_values("max_force_vector_error_eV_A", ascending=False).head(10)
    display(worst)
    del ft_calc
    torch.cuda.empty_cache()
else:
    print("Skipped. Train first, then set RUN_FINE_TUNED_EVAL=True.")

Skipped. Train first, then set RUN_FINE_TUNED_EVAL=True.


## 11. Validate the potential as physics

An MLIP is a dynamical model, not a table of test errors. Before scientific production, validate at least:

### Static checks

- energy and force residuals by composition, coordination, adsorbate, facet, temperature, and active-learning round;
- adsorption/reaction/defect energies with the same reference convention as DFT;
- relaxed geometries, bond lengths, lattice constants, and relative ordering of states;
- predicted-versus-reference stress if the cell will move.

### Dynamical checks

- short NVE energy drift with a conservative timestep;
- NVT/NPT stability over the intended temperature/pressure range;
- no unphysical atom overlap, desorption, fragmentation, or force spikes;
- uncertainty/extrapolation signals on frames visited during MD;
- DFT spot checks harvested from the deployed trajectory.

### Transfer checks

- hold out complete chemical motifs or trajectories;
- evaluate the original domain as well as the new target domain;
- test compressed, high-force, and reaction-like structures;
- declare the validated scope. “Works on Pt(111) + O/OH/OOH near equilibrium” is a useful statement; “universal catalyst model” usually is not.

**Go/no-go principle:** publish or deploy only after both held-out DFT metrics and target physical observables pass predeclared criteria.

## 12. Iterative fine-tuning: the Kwon–Graves strategy

The attached 2026 WS₂ oxygen-plasma paper used a stronger loop than one-shot fine-tuning:

1. run MLIP-driven exploration/MD;
2. represent environments with SOAP descriptors;
3. reduce descriptor dimension with PCA (50 components retained about 99% variance in that study);
4. select diverse structures by farthest-point sampling;
5. label them with consistent spin-polarized PBE+D3+U DFT;
6. fine-tune on the cumulative data;
7. evaluate on newly generated next-round structures that the model has never seen;
8. repeat until metrics **and physical observables** converge.

Their final three-round model used 2,610 configurations and reported about 4.5 meV/atom energy MAE, 0.076 eV/Å force MAE, and 0.034 GPa stress MAE for the study's held-out distribution. These numbers are context-specific, not general UMA guarantees.

The key lesson was that data coverage—not merely the number of trainable parameters—was the bottleneck. Full fine-tuning, layer freezing, and LoRA became similar once the dataset covered the relevant configurations.

In [16]:
def farthest_point_sampling(features, n_select, seed=7):
    '''Greedy max-min selection on scaled/PCA-projected configuration descriptors.'''
    x = np.asarray(features, dtype=float)
    if x.ndim != 2 or len(x) == 0:
        raise ValueError("features must have shape (n_configurations, n_features)")
    n_select = min(int(n_select), len(x))
    rng = np.random.default_rng(seed)
    selected = [int(rng.integers(len(x)))]
    min_sq_distance = np.sum((x - x[selected[0]]) ** 2, axis=1)
    for _ in range(1, n_select):
        next_index = int(np.argmax(min_sq_distance))
        selected.append(next_index)
        new_sq_distance = np.sum((x - x[next_index]) ** 2, axis=1)
        min_sq_distance = np.minimum(min_sq_distance, new_sq_distance)
    return np.asarray(selected, dtype=int)


# Example after building one descriptor vector per candidate configuration:
# from sklearn.preprocessing import StandardScaler
# from sklearn.decomposition import PCA
# x_scaled = StandardScaler().fit_transform(configuration_descriptors)
# n_pc = min(50, x_scaled.shape[0] - 1, x_scaled.shape[1])
# x_pca = PCA(n_components=n_pc, random_state=7).fit_transform(x_scaled)
# selected_indices = farthest_point_sampling(x_pca, n_select=100)
# write("selected_for_dft.extxyz", [candidate_frames[i] for i in selected_indices])

print("FPS helper ready. Descriptor construction is chemistry-dependent; inspect selected structures before DFT.")

FPS helper ready. Descriptor construction is chemistry-dependent; inspect selected structures before DFT.


### Active-learning cautions

- A mean-pooled structure descriptor can hide a rare but dangerous local environment. For reactions, select in local-environment space or combine global and maximum/quantile summaries.
- Diversity is not the same as model error. Combine diversity selection with disagreement/uncertainty and physical filters when possible.
- Do not label obvious duplicates, but do not delete genuine high-force or transition-like frames merely because they look unusual.
- Never let test labels flow into model selection. Once you add a next-round test to cumulative training, generate a **fresh** test set for the following round.

## 13. Legacy OCP/GemNet-OC → current Fair-Chem UMA migration

Your attached notes contain both generations. They are useful historically, but should not be combined in one runnable pipeline.

| Legacy Fair-Chem v1 / OCP tutorial | Current Fair-Chem v2 / UMA |
|---|---|
| GemNet-OC/SCN checkpoints from old registry | UMA model name such as `uma-s-1p2` |
| `OCPCalculator` | `FAIRChemCalculator` |
| `main.py` / `flags` / `build_config` / trainer context | `fairchem -c CONFIG.yaml` with Hydra overrides |
| manual split/config utilities | labelled folders → `create_uma_finetune_dataset.py` |
| old LMDB/OCP trainer assumptions | ASE-LMDB and UMA data tasks |
| TensorBoard examples | current fine-tuning docs use Weights & Biases; `job.debug=True` for no logger |
| continue a particular GemNet-OC architecture | pretrained UMA backbone + newly initialized target head |

Use Fair-Chem `v1.10.0` only when reproducing a legacy checkpoint/workflow. Do not install v1 and expect the v2 notebook API to work.

## 14. A 60–90 minute teaching plan

### Before class

- obtain gated model access;
- enable a suitable GPU;
- run setup and the official data conversion once;
- prepare three labelled examples: equilibrium, distorted, and an obvious outlier.

### In class

1. **10 min — Concepts:** pretraining, task heads, label theory, and the new-head reset.
2. **15 min — Data:** inspect ASE labels, units, constraints, and group-aware splits.
3. **10 min — Audit:** identify bad labels and leakage in a deliberately flawed dataset.
4. **15 min — Pipeline:** run converter, inspect energy references/force RMS, read Hydra YAML.
5. **10 min — Training:** launch smoke run and interpret logs/checkpoints.
6. **15 min — Evaluation:** compare zero-shot/fine-tuned errors and worst cases.
7. **10 min — Research design:** sketch an active-learning round and validation gates.

### Exit questions

1. Why can fine-tuning be worse than zero-shot at step 0?
2. Why is a random frame split unsafe for MD data?
3. When would `efs` be justified rather than `ef`?
4. What does a low force MAE fail to prove?
5. When should you add a configuration to training, and when should it remain a test case?

## 15. Reproducibility record and export

Save the resolved config, repository commit, model/task name, raw-data manifest, split logic, DFT settings, seeds, package versions, checkpoint, and evaluation tables. Without these, “fine-tuned UMA” is not a reproducible method.

In [17]:
record = {
    "fairchem_git_ref": FAIRCHEM_REF,
    "base_model": BASE_MODEL,
    "uma_task": UMA_TASK,
    "regression_tasks": REGRESSION_TASKS,
    "dataset_tag": DATASET_TAG,
    "run_id": RUN_ID,
    "device": DEVICE,
    "torch": torch.__version__,
    "train_frames": len(train_audit),
    "validation_frames": len(val_audit),
    "test_frames": len(test_audit),
}

record_path = WORK / "reproducibility_record.json"
record_path.write_text(json.dumps(record, indent=2))
print(record_path.read_text())

# Optional: download the record and final checkpoint from Colab.
# from google.colab import files
# files.download(str(record_path))
# files.download(str(INFERENCE_CKPT))

{
  "fairchem_git_ref": "e4425dcbd6649b36a82bc513585b8664fb7a6b0a",
  "base_model": "uma-s-1p2",
  "uma_task": "omat",
  "regression_tasks": "e",
  "dataset_tag": "demo_v1",
  "run_id": "uma_ft_smoke_v1",
  "device": "cuda",
  "torch": "2.13.0+cu130",
  "train_frames": 2,
  "validation_frames": 2,
  "test_frames": 2
}


## References and further reading

Primary sources used for this tutorial:

- [Current Fair-Chem UMA fine-tuning guide](https://github.com/facebookresearch/fairchem/blob/main/docs/core/common_tasks/fine_tuning.md)
- [Current dataset-conversion source](https://github.com/facebookresearch/fairchem/blob/main/src/fairchem/core/scripts/create_uma_finetune_dataset.py)
- [Fine-tuning template configuration](https://github.com/facebookresearch/fairchem/blob/main/configs/uma/finetune/uma_sm_finetune_template.yaml)
- [Fair-Chem repository and v1/v2 migration note](https://github.com/facebookresearch/fairchem)
- [UMA paper](https://arxiv.org/abs/2506.23971)
- [Maintainer explanation of reinitialized heads](https://github.com/facebookresearch/fairchem/issues/1486)
- [Kwon & Graves: iterative UMA fine-tuning for oxygen-plasma/WS₂](https://arxiv.org/abs/2606.21632)

### Suggested exercise

Duplicate the notebook and replace the official demo with one small, internally consistent DFT dataset. Before training, write down:

- intended domain and UMA task;
- DFT theory contract;
- group used for splitting;
- error and physical-observable acceptance criteria;
- configurations intentionally outside the model's claimed scope.

Then compare zero-shot and fine-tuned results without changing the test set.